Imports

In [ ]:
# installing libraries used for the project
%pip install pandas numpy scikit-learn lightgbm torch shap lime alibi dice-ml
%pip install "xgboost==3.0.5"

In [121]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, f1_score
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import shap
from lime.lime_tabular import LimeTabularExplainer
from alibi.explainers import AnchorTabular
import dice_ml
from dice_ml import Dice
import pickle
import os
#ignore warnings
import warnings
warnings.filterwarnings("ignore")

In [122]:
#path of dataset files and artifacts
train_data_path = "../Datasets/fraudTrainData.csv"
test_data_path  = "../Datasets/fraudTestData.csv"

model_bundle_path = "../Artifacts/model_bundle.pkl"
nn_model_path = "../Artifacts/nn_model.pt"

Loading datasets

In [123]:
train_df = pd.read_csv(train_data_path)
test_df  = pd.read_csv(test_data_path)

print("Train data shape:", train_df.shape)
print("Test data shape :", test_df.shape)  

Train data shape: (1296675, 23)
Test data shape : (555719, 23)


In [124]:
# Compare columns in train and test data
print("Columns only in train:")
print(set(train_df.columns) - set(test_df.columns))

print("\nColumns only in test:")
print(set(test_df.columns) - set(train_df.columns))

# Analyze target variable distribution
target_col = "is_fraud"
print("\nTarget distribution in train data:")
print(train_df[target_col].value_counts())
print("\nFraud ratio:")
print(train_df[target_col].value_counts(normalize=True)*100)

# Check missing values
print("\nMissing values")
print(train_df.isnull().sum())

Columns only in train:
set()

Columns only in test:
set()

Target distribution in train data:
is_fraud
0    1289169
1       7506
Name: count, dtype: int64

Fraud ratio:
is_fraud
0    99.421135
1     0.578865
Name: proportion, dtype: float64

Missing values
Unnamed: 0               0
trans_date_trans_time    0
cc_num                   0
merchant                 0
category                 0
amt                      0
first                    0
last                     0
gender                   0
street                   0
city                     0
state                    0
zip                      0
lat                      0
long                     0
city_pop                 0
job                      0
dob                      0
trans_num                0
unix_time                0
merch_lat                0
merch_long               0
is_fraud                 0
dtype: int64


Drop unwanted columns

In [125]:
# Columns to drop (identifiers, PII, leakage)
drop_cols = [       
    "Unnamed: 0", 
    "cc_num", 
    "merchant",
    "first", "last",  
    "street", "city", "state", "zip",  
    "trans_num",      
    "unix_time",      
    "job"             
]

# Datasets after dropping unwanted columns
X_train = train_df.drop(columns=drop_cols + ["is_fraud"])
y_train = train_df["is_fraud"]
print("\nTrain data shape after dropping columns:", X_train.shape)

X_test = test_df.drop(columns=drop_cols + ["is_fraud"])
y_test = test_df["is_fraud"]
print("Test data shape after dropping columns :", X_test.shape)

print("\nRemaining feature columns:")
print(X_train.columns.tolist())
print("\nNumber of features:", X_train.shape[1])



Train data shape after dropping columns: (1296675, 10)
Test data shape after dropping columns : (555719, 10)

Remaining feature columns:
['trans_date_trans_time', 'category', 'amt', 'gender', 'lat', 'long', 'city_pop', 'dob', 'merch_lat', 'merch_long']

Number of features: 10


Feature engineering

In [126]:
def add_time_and_age_features(df):
    df = df.copy()

    # Parse datetime & dob
    df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"])
    df["dob"] = pd.to_datetime(df["dob"])

    # Extract hour, month, weekend
    df["trans_hour"] = df["trans_date_trans_time"].dt.hour
    df["trans_month"] = df["trans_date_trans_time"].dt.month
    df["is_weekend"] = (df["trans_date_trans_time"].dt.dayofweek >= 5).astype(int)
    
    # Age in years
    age_days = (df["trans_date_trans_time"] - df["dob"]).dt.days
    df["age"] = (age_days / 365.25).astype(int)

    # Drop original datetime and dob columns
    df.drop(columns=["trans_date_trans_time", "dob"], inplace=True)

    return df

X_train_fe = add_time_and_age_features(X_train)
X_test_fe  = add_time_and_age_features(X_test)

print("\nFeatures after datetime processing:")
print(X_train_fe.columns.tolist())
print(X_test_fe.columns.tolist())


Features after datetime processing:
['category', 'amt', 'gender', 'lat', 'long', 'city_pop', 'merch_lat', 'merch_long', 'trans_hour', 'trans_month', 'is_weekend', 'age']
['category', 'amt', 'gender', 'lat', 'long', 'city_pop', 'merch_lat', 'merch_long', 'trans_hour', 'trans_month', 'is_weekend', 'age']


In [127]:
# Seperating categorical and numeric columns
categorical_cols = ["category", "gender"]
numeric_cols = [col for col in X_train_fe.columns if col not in categorical_cols]

# ColumnTransformer to combine scale numeric, one-hot encode categorical
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols)
    ]
)

X_train_processed = preprocessor.fit_transform(X_train_fe)
X_test_processed  = preprocessor.transform(X_test_fe)

print("Processed train shape:", X_train_processed.shape)
print("Processed test shape :", X_test_processed.shape)


Processed train shape: (1296675, 26)
Processed test shape : (555719, 26)


Convert the data to dataframes

In [128]:
feature_names = preprocessor.get_feature_names_out()

X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=feature_names
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=feature_names
)
feature_names=list(feature_names)
print("Train columns:", X_train_processed.shape)
print("Test columns :", X_test_processed.shape)

Train columns: (1296675, 26)
Test columns : (555719, 26)


Training ML Models

In [129]:
# ML model training (RF + XGB + LGBM)

rf_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    class_weight="balanced_subsample",       
    n_jobs=-1,
    random_state=42
)

xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=5,                             # less depth less overfitting
    learning_rate=0.05,                      
    subsample=0.8,
    colsample_bytree=0.7,
    min_child_weight=5,                      
    gamma=1.0,                               
    reg_alpha=0.1,                           
    reg_lambda=1.0,                          
    scale_pos_weight=10,                     
    eval_metric="aucpr",
    n_jobs=-1,
    random_state=42,
    tree_method="hist"                       # quick on large datasets
)

lgbm_model = LGBMClassifier(
    n_estimators=600,
    learning_rate=0.05,
    num_leaves=63,                           
    max_depth=8,
    min_child_samples=50,                    
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.7,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=10,                     
    is_unbalance=False,                      # use scale_pos_weight only
    n_jobs=-1,
    random_state=42,
    verbose=-1
)

In [130]:
# Train models
print("Training RandomForest...")
rf_model.fit(X_train_processed, y_train)

print("Training XGBoost...")
xgb_model.fit(X_train_processed, y_train)

print("Training LightGBM...")
lgbm_model.fit(X_train_processed, y_train)

print("\n Training completed!")

# Evaluation helper function
def evaluate_classification_model(model, threshold, X_test, y_test, name="Model"):
    proba = model.predict_proba(X_test)[:, 1]
    preds = (proba >= threshold).astype(int)
    roc_auc = roc_auc_score(y_test, proba)
    pr_auc = average_precision_score(y_test, proba)

    print(f"\n===== {name} =====")
    print("ROC-AUC:", round(roc_auc, 4))
    print("PR-AUC :", round(pr_auc, 4))
    print(classification_report(y_test, preds, digits=4))

# Evaluate each
evaluate_classification_model(rf_model, 0.3, X_test_processed, y_test, "RandomForest")
evaluate_classification_model(xgb_model, 0.6, X_test_processed, y_test, "XGBoost")
evaluate_classification_model(lgbm_model, 0.6, X_test_processed, y_test, "LightGBM")


Training RandomForest...
Training XGBoost...
Training LightGBM...

 Training completed!

===== RandomForest =====
ROC-AUC: 0.9892
PR-AUC : 0.8769
              precision    recall  f1-score   support

           0     0.9992    0.9996    0.9994    553574
           1     0.8979    0.7953    0.8435      2145

    accuracy                         0.9989    555719
   macro avg     0.9486    0.8975    0.9215    555719
weighted avg     0.9988    0.9989    0.9988    555719


===== XGBoost =====
ROC-AUC: 0.9974
PR-AUC : 0.872
              precision    recall  f1-score   support

           0     0.9992    0.9992    0.9992    553574
           1     0.7888    0.8042    0.7964      2145

    accuracy                         0.9984    555719
   macro avg     0.8940    0.9017    0.8978    555719
weighted avg     0.9984    0.9984    0.9984    555719


===== LightGBM =====
ROC-AUC: 0.9968
PR-AUC : 0.8607
              precision    recall  f1-score   support

           0     0.9991    0.9995    0.

MLP using pytorch

In [131]:
X_train_t = torch.tensor(X_train_processed.to_numpy(), dtype=torch.float32)
Y_train_t = torch.tensor(y_train.to_numpy(), dtype=torch.float32).view(-1, 1)

X_test_t = torch.tensor(X_test_processed.to_numpy(), dtype=torch.float32)
Y_test_t = torch.tensor(y_test.to_numpy(), dtype=torch.float32).view(-1, 1)

train_loader = DataLoader(TensorDataset(X_train_t, Y_train_t), batch_size=4096, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test_t, Y_test_t), batch_size=4096, shuffle=False)


In [132]:
class FraudMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)

input_dim = X_train_processed.shape[1]
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
mlp_model = FraudMLP(input_dim=input_dim).to(device)

Device: cpu


In [133]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
pos_weight_raw = neg / pos                   # ≈ 172
pos_weight_capped = torch.tensor(
    [min(float(pos_weight_raw), 15.0)],      # cap at 15
    dtype=torch.float32
).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_capped)

optimizer = torch.optim.AdamW(
    mlp_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
)


In [134]:
def train_mlp_model(model, train_loader, epochs=10):
    model.train()
    for ep in range(1, epochs + 1):
        total_loss = 0.0
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)

            optimizer.zero_grad()
            logits = model(x_batch)
            loss   = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * x_batch.size(0)

        avg_loss = total_loss / len(train_loader.dataset)
        print(f"Epoch {ep}/{epochs} | Loss: {avg_loss:.4f}")
        scheduler.step(avg_loss)            

train_mlp_model(mlp_model, train_loader, epochs=10)

Epoch 1/10 | Loss: 0.1545
Epoch 2/10 | Loss: 0.0940
Epoch 3/10 | Loss: 0.0764
Epoch 4/10 | Loss: 0.0641
Epoch 5/10 | Loss: 0.0583
Epoch 6/10 | Loss: 0.0541
Epoch 7/10 | Loss: 0.0509
Epoch 8/10 | Loss: 0.0488
Epoch 9/10 | Loss: 0.0484
Epoch 10/10 | Loss: 0.0465


In [135]:
# MLP Evaluation
mlp_model.eval()
all_probs = []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        yb = yb.cpu().numpy().ravel()  

        logits = mlp_model(xb)          # shape: [batch, 1]
        probs  = torch.sigmoid(logits).cpu().numpy().ravel()

        all_probs.append(probs)

mlp_probs = np.concatenate(all_probs)

mlp_pred = (mlp_probs >= 0.7).astype(int)

roc_auc = roc_auc_score(y_test, mlp_probs)
pr_auc  = average_precision_score(y_test, mlp_probs)
print("\n===== MLP =====")
print("ROC-AUC:", round(roc_auc, 4))
print("PR-AUC :", round(pr_auc, 4))
print(classification_report(y_test, mlp_pred, digits=4))


===== MLP =====
ROC-AUC: 0.9949
PR-AUC : 0.8368
              precision    recall  f1-score   support

           0     0.9993    0.9984    0.9988    553574
           1     0.6591    0.8168    0.7295      2145

    accuracy                         0.9977    555719
   macro avg     0.8292    0.9076    0.8642    555719
weighted avg     0.9980    0.9977    0.9978    555719



In [136]:
# Probabilities from ML models
rf_probs   = rf_model.predict_proba(X_test_processed)[:, 1]
xgb_probs  = xgb_model.predict_proba(X_test_processed)[:, 1]
lgbm_probs = lgbm_model.predict_proba(X_test_processed)[:, 1]

print("Shapes:", rf_probs.shape, xgb_probs.shape, lgbm_probs.shape, mlp_probs.shape)


Shapes: (555719,) (555719,) (555719,) (555719,)


WEIGHTED ENSEMBLE

In [137]:
# weights must sum to 1
w_rf, w_xgb, w_lgbm, w_mlp = 0.30, 0.35, 0.20, 0.15

ensemble_prediction_probability = (
    w_rf  * rf_probs +
    w_xgb * xgb_probs +
    w_lgbm* lgbm_probs +
    w_mlp * mlp_probs
)

thresholds = np.linspace(0.05, 0.95, 50)
best_t, best_f1 = 0.5, -1

for t in thresholds:
    pred = (ensemble_prediction_probability >= t).astype(int)
    f1 = f1_score(y_test, pred, pos_label=1)   # fraud class F1
    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print("Best threshold:", round(best_t, 3))
print("Best F1:", round(best_f1, 4))

best_pred = (ensemble_prediction_probability >= best_t).astype(int)

roc_auc = roc_auc_score(y_test, ensemble_prediction_probability)
pr_auc  = average_precision_score(y_test, ensemble_prediction_probability)

print("\n===== ENSEMBLE (Weighted) =====")
print("ROC-AUC:", round(roc_auc, 4))
print("PR-AUC :", round(pr_auc, 4))
print("Threshold:", round(best_t, 3))
print(classification_report(y_test, best_pred, digits=4))

Best threshold: 0.601
Best F1: 0.8395

===== ENSEMBLE (Weighted) =====
ROC-AUC: 0.9973
PR-AUC : 0.8843
Threshold: 0.601
              precision    recall  f1-score   support

           0     0.9991    0.9997    0.9994    553574
           1     0.9134    0.7767    0.8395      2145

    accuracy                         0.9989    555719
   macro avg     0.9563    0.8882    0.9195    555719
weighted avg     0.9988    0.9989    0.9988    555719



In [138]:
ensemble_confidence = np.where(
    best_pred == 1,
    ensemble_prediction_probability,
    1 - ensemble_prediction_probability
)
print("Confidence sample average:", ensemble_confidence.mean())


Confidence sample average: 0.995207094563632


Implementing XAI models

In [139]:
# MLP probability helper (processed DF with same columns as X_train_processed)
def predict_mlp_probabilities(X_df):
    mlp_model.eval()
    X_t = torch.tensor(X_df.to_numpy(), dtype=torch.float32).to(device)
    with torch.no_grad():
        p = torch.sigmoid(mlp_model(X_t)).cpu().numpy().ravel()
    return np.vstack([1 - p, p]).T

# Ensemble probability helper (processed DF with same columns as X_train_processed)
def predict_ensemble_probabilities(X_processed_df):
    rf_p   = rf_model.predict_proba(X_processed_df)[:, 1]
    xgb_p  = xgb_model.predict_proba(X_processed_df)[:, 1]
    lgbm_p = lgbm_model.predict_proba(X_processed_df)[:, 1]
    mlp_p  = predict_mlp_probabilities(X_processed_df)[:, 1]
    ens_p = (w_rf*rf_p + w_xgb*xgb_p + w_lgbm*lgbm_p + w_mlp*mlp_p)
    return np.vstack([1 - ens_p, ens_p]).T

def predict_ensemble_labels(X_processed_df):
    X_df = pd.DataFrame(X_processed_df, columns=feature_names)
    p = predict_ensemble_probabilities(X_df)[:, 1]
    return (p >= float(best_t)).astype(int)


SHAP

In [140]:
# Background sample
X_bg = X_train_processed.sample(500, random_state=42)

# Tree explainers 
shap_rf   = shap.TreeExplainer(rf_model,  data=X_bg)
shap_xgb  = shap.TreeExplainer(xgb_model, data=X_bg)
shap_lgbm = shap.TreeExplainer(lgbm_model, data=X_bg)

# MLP Kernel explainer (smaller bg) 
X_bg_small = X_bg.sample(300, random_state=42).to_numpy()

def mlp_predict_proba_from_numpy(X_np):
    X_df = pd.DataFrame(X_np, columns=list(feature_names))
    return predict_mlp_probabilities(X_df)

shap_mlp = shap.KernelExplainer(lambda x: mlp_predict_proba_from_numpy(x)[:, 1], X_bg_small)


Using 300 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


In [141]:
def get_absolute_shap_for_fraud_class(explainer, X_df, class_idx=1):
  
    sv = explainer.shap_values(X_df)

    # list - pick fraud class
    if isinstance(sv, list):
        sv = sv[class_idx]

    sv = np.array(sv)

    # (1, n_features) -> take [0]
    if sv.ndim == 2 and sv.shape[0] == 1:
        vec = sv[0]

    # (n_features, 2) -> pick class column
    elif sv.ndim == 2 and sv.shape[1] == 2:
        vec = sv[:, class_idx]

    # (1, n_features, 2) -> take [0,:,class]
    elif sv.ndim == 3 and sv.shape[0] == 1 and sv.shape[-1] == 2:
        vec = sv[0, :, class_idx]

    # (n_features,) already
    elif sv.ndim == 1:
        vec = sv

    else:
        # fallback: try squeezing
        sv_s = np.squeeze(sv)
        if sv_s.ndim == 2 and sv_s.shape[1] == 2:
            vec = sv_s[:, class_idx]
        elif sv_s.ndim == 1:
            vec = sv_s
        else:
            raise ValueError(f"Unexpected SHAP shape: {sv.shape}")

    return np.abs(vec).astype(float)


def get_per_model_shap_single_row(X_proc_df, nsamples_mlp=200):
    """
    Returns dict of abs shap arrays for ONE processed row in X_proc_df, for each model.
    """
    if len(X_proc_df) != 1:
        raise ValueError("X_proc_df must be a single-row dataframe.")

    # enforce column order
    X_proc_df = X_proc_df[feature_names].copy()

    rf_abs   = get_absolute_shap_for_fraud_class(shap_rf,   X_proc_df, class_idx=1)
    xgb_abs  = get_absolute_shap_for_fraud_class(shap_xgb,  X_proc_df, class_idx=1)
    lgbm_abs = get_absolute_shap_for_fraud_class(shap_lgbm, X_proc_df, class_idx=1)

    # KernelExplainer output can be list or array
    mlp_sv = shap_mlp.shap_values(X_proc_df.to_numpy(), nsamples=nsamples_mlp)
    if isinstance(mlp_sv, list):
        mlp_sv = mlp_sv[0]
    mlp_sv = np.array(mlp_sv)

    # make it (n_features,)
    mlp_sv = np.squeeze(mlp_sv)
    if mlp_sv.ndim == 2 and mlp_sv.shape[0] == 1:
        mlp_sv = mlp_sv[0]
    mlp_abs = np.abs(mlp_sv).astype(float)

    return {"rf": rf_abs, "xgb": xgb_abs, "lgbm": lgbm_abs, "mlp": mlp_abs}


def fuse_weighted_ensemble_shap(per_model_shap_abs, top_k=10):
    weights = {"rf": w_rf, "xgb": w_xgb, "lgbm": w_lgbm, "mlp": w_mlp}
    fused = np.zeros(len(feature_names), dtype=float)

    for k, arr in per_model_shap_abs.items():
        arr = np.array(arr, dtype=float)
        s = arr.sum()
        arr_norm = arr / s if s > 0 else arr
        fused += weights[k] * arr_norm

    df = pd.DataFrame({"feature": feature_names, "ensemble_shap_score": fused})
    return df.sort_values("ensemble_shap_score", ascending=False).head(top_k).reset_index(drop=True)


LIME

In [142]:
# LIME training data
lime_train = X_train_processed.sample(500, random_state=42).to_numpy()

lime_explainer = LimeTabularExplainer(
    training_data=lime_train,
    feature_names=feature_names,      
    class_names=["genuine", "fraud"],
    mode="classification",
    discretize_continuous=True
)

# feature mapping 
def map_lime_rule_to_feature_name(rule):
    for f in feature_names:
        if f in rule:
            return f
    return rule

# Explain ONE processed row (ensemble output) and return fusion-ready table
def explain_single_instance_with_lime_ensemble(X_one_df, top_k=10):
    # single row + correct column order
    if len(X_one_df) != 1:
        raise ValueError("X_one_df must be a single-row dataframe.")
    X_one_df = X_one_df[feature_names].copy()

    exp = lime_explainer.explain_instance(
        data_row=X_one_df.to_numpy().ravel(),
        predict_fn=lambda x: predict_ensemble_probabilities(
            pd.DataFrame(x, columns=feature_names)
        ),
        num_features=top_k
    )

    rows = []
    for rule, w in exp.as_list():
        rows.append({
            "feature": map_lime_rule_to_feature_name(rule),
            "rule": rule,
            "lime_score": abs(w)
        })

    df = pd.DataFrame(rows).sort_values("lime_score", ascending=False).reset_index(drop=True)
    return df, exp


ANCHOR

In [143]:
anchor_bg = X_train_processed.sample(500, random_state=42).to_numpy()

anchor_explainer = AnchorTabular(
    predictor=predict_ensemble_labels,
    feature_names=list(feature_names),
    seed=42
)
anchor_explainer.fit(anchor_bg, disc_perc=(25, 50, 75))


def get_anchor_explanation_for_single_instance(X_df, threshold=0.95):
    exp = anchor_explainer.explain(X_df.to_numpy()[0], threshold=threshold, seed=42)
    anchor_list = exp.data.get("anchor", [])
    anchor_rule = " AND ".join(anchor_list).strip()
    return {
        "anchor_rule": anchor_rule if anchor_rule else None,
        "precision": float(exp.data["precision"]),
        "coverage": float(exp.data["coverage"]),
        "threshold": float(threshold)
    }
    


DiCE

In [144]:
# DiCE works in raw fe space (X_train_fe)
class EnsembleDiceWrapper:
    def __init__(self, preprocessor, processed_feature_names):
        self.preprocessor = preprocessor
        self.processed_feature_names = list(processed_feature_names)

    def predict_proba(self, X_raw_df):
        # X_raw_df is in fe raw space (same cols as X_train_fe)
        X_proc = self.preprocessor.transform(X_raw_df)
        X_proc_df = pd.DataFrame(X_proc, columns=self.processed_feature_names)
        return predict_ensemble_probabilities(X_proc_df)

    def predict(self, X_raw_df):
        p = self.predict_proba(X_raw_df)[:, 1]
        return (p >= float(best_t)).astype(int)

dice_model = EnsembleDiceWrapper(preprocessor, feature_names)

# DiCE Data object in raw fe space
dice_continuous = [c for c in X_train_fe.columns if c not in categorical_cols]
dice_categorical = list(categorical_cols)

dice_train_df = X_train_fe.copy()
dice_train_df["is_fraud"] = y_train.values

dice_data = dice_ml.Data(
    dataframe=dice_train_df,
    continuous_features=dice_continuous,
    categorical_features=dice_categorical,
    outcome_name="is_fraud"
)

dice_model_obj = dice_ml.Model(model=dice_model, backend="sklearn", model_type="classifier")

# random is fastest for prototype
dice_explainer = Dice(dice_data, dice_model_obj, method="random")

print("DiCE explainer is ready")
print("Continuous features:", dice_continuous)
print("Categorical features:", dice_categorical)


DiCE explainer is ready
Continuous features: ['amt', 'lat', 'long', 'city_pop', 'merch_lat', 'merch_long', 'trans_hour', 'trans_month', 'is_weekend', 'age']
Categorical features: ['category', 'gender']


In [145]:
# explain ONE row in raw fe space 
permitted_range = {
    "amt": [0, X_train_fe["amt"].quantile(0.99)],
    "age": [18, 100],
    "trans_hour": [0, 23],
    "trans_month": [1, 12],
    "is_weekend": [0, 1],
}

def get_dice_counterfactuals_for_single_instance(X_one_raw_fe_df, total_CFs=3, desired_class="opposite", features_to_vary="all"):
    """
    X_one_raw_fe_df: 1-row dataframe in RAW feature-engineered space (same columns as X_train_fe)
    Returns:
      cf_df: DataFrame of counterfactuals (raw FE space)
      exp  : DiCE explanation object
    """
    if not isinstance(X_one_raw_fe_df, pd.DataFrame):
        raise ValueError("X_one_raw_fe_df must be a pandas DataFrame.")
    if len(X_one_raw_fe_df) != 1:
        raise ValueError("X_one_raw_fe_df must be a SINGLE-row dataframe.")

    # Ensure correct column order (important for DiCE consistency)
    X_one_raw_fe_df = X_one_raw_fe_df[X_train_fe.columns].copy()

    exp = dice_explainer.generate_counterfactuals(
        query_instances=X_one_raw_fe_df,
        total_CFs=total_CFs,
        desired_class=desired_class,
        features_to_vary=[
            "amt", "category", "trans_hour", "is_weekend"
        ],
        permitted_range=permitted_range
    )

    # Extract counterfactuals dataframe 
    try:
        cf_df = exp.cf_examples_list[0].final_cfs_df.copy()
        cf_df = cf_df[X_train_fe.columns]
    except Exception:
        cf_df = pd.DataFrame(columns=X_train_fe.columns)

    return cf_df, exp


In [146]:
def map_onehot_to_base_feature(f):
    # remove "num__" and "cat__"
    f = str(f).replace("num__", "").replace("cat__", "")
    if f.startswith("category_"):
        return "category"
    if f.startswith("gender_"):
        return "gender"
    return f

def ensemble_feature_importance(df, feature_col, score_col, k=10):
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=["feature", score_col])
    out = df[[feature_col, score_col]].copy()
    out["feature"] = out[feature_col].apply(map_onehot_to_base_feature)
    out = out.groupby("feature", as_index=False)[score_col].sum()
    out = out.sort_values(score_col, ascending=False).head(k).reset_index(drop=True)
    return out[["feature", score_col]]

def summarize_counterfactual_changes(original_raw_fe_df, cf_df, max_cfs=3):
    if cf_df is None or len(cf_df) == 0:
        return []
    orig = original_raw_fe_df.iloc[0]
    summaries = []
    for idx in range(min(max_cfs, len(cf_df))):
        cf = cf_df.iloc[idx]
        changed = []
        for c in original_raw_fe_df.columns:
            if c in cf.index and str(orig[c]) != str(cf[c]):
                changed.append({"feature": c, "from": orig[c], "to": cf[c]})
        summaries.append({"cf_id": idx + 1, "changes": changed})
    return summaries

def explain_transaction_with_ensemble_xai(raw_input_df,
                           top_k=8,
                           dice_total_cfs=3,
                           anchor_threshold=0.95,
                           mlp_nsamples=150,
                           w_shap=0.6,
                           w_lime=0.4):
    """
    Returns: dict with prediction + combined explanations (SHAP+LIME+Anchors+DiCE)
    """

    if not isinstance(raw_input_df, pd.DataFrame) or len(raw_input_df) != 1:
        raise ValueError("raw_input_df must be a SINGLE-row pandas DataFrame.")

    # Feature engineering (RAW -> FE)
    raw_fe = add_time_and_age_features(raw_input_df)
    raw_fe = raw_fe[X_train_fe.columns].copy()  # enforce order

    # Preprocess (FE -> processed)
    X_proc = preprocessor.transform(raw_fe)
    X_proc_df = pd.DataFrame(X_proc, columns=feature_names)

    # Prediction
    ens_proba = float(predict_ensemble_probabilities(X_proc_df)[0, 1])
    ens_label = int(ens_proba >= float(best_t))
    confidence = float(ens_proba if ens_label == 1 else 1 - ens_proba)

    # SHAP per-model -> fused SHAP
    per_model_shap = get_per_model_shap_single_row(X_proc_df, nsamples_mlp=mlp_nsamples)
    shap_fused_df = fuse_weighted_ensemble_shap(per_model_shap, top_k=len(feature_names))  # get all, then collapse/group
    shap_fused_df = shap_fused_df.rename(columns={"ensemble_shap_score": "shap_score"})
    shap_top = ensemble_feature_importance(shap_fused_df, "feature", "shap_score", k=top_k)

    # LIME
    lime_df, lime_exp = explain_single_instance_with_lime_ensemble(X_proc_df, top_k=top_k)
    lime_top = ensemble_feature_importance(lime_df, "feature", "lime_score", k=top_k)

    # Anchors
    anchor_output = get_anchor_explanation_for_single_instance(X_proc_df, threshold=anchor_threshold)

    # DiCE (RAW FE space)
    cf_df, dice_exp = get_dice_counterfactuals_for_single_instance(raw_fe, total_CFs=dice_total_cfs, desired_class="opposite")
    dice_summary = summarize_counterfactual_changes(raw_fe, cf_df, max_cfs=dice_total_cfs)

    # Final fused explanation (SHAP + LIME)
    shap_norm = shap_top.copy()
    lime_norm = lime_top.copy()

    shap_norm["shap_score"] = shap_norm["shap_score"] / (shap_norm["shap_score"].sum() + 1e-9)
    lime_norm["lime_score"] = lime_norm["lime_score"] / (lime_norm["lime_score"].sum() + 1e-9)

    merged = shap_norm.merge(lime_norm, on="feature", how="outer").fillna(0.0)
    merged["ensemble_expl_score"] = (w_shap * merged["shap_score"]) + (w_lime * merged["lime_score"])
    merged = merged.sort_values("ensemble_expl_score", ascending=False).head(top_k).reset_index(drop=True)

    return {
        "prediction": "fraud" if ens_label == 1 else "genuine",
        "fraud_probability": ens_proba,
        "confidence": confidence,
        "threshold_used": float(best_t),

        "explanations": {
            "final_ensemble_summary": merged.to_dict(orient="records"),   
            "shap_top": shap_top.to_dict(orient="records"),
            "lime_top": lime_top.to_dict(orient="records"),
            "anchor": anchor_output,
            "dice": {
                "total_found": int(len(cf_df)) if cf_df is not None else 0,
                "counterfactuals": cf_df,            
                "changes_summary": dice_summary
            }
        }
    }


In [147]:
raw_one = test_df.iloc[[10]].drop(columns=drop_cols + ["is_fraud"], errors="ignore")
out = explain_transaction_with_ensemble_xai(raw_one)
print("Prediction:" + out["prediction"] + "\nFraud Probability: " + str(out["fraud_probability"]) + "\nConfidence: " + str(out["confidence"]))
print(out["explanations"]["final_ensemble_summary"])
print(out["explanations"]["anchor"])
print(out["explanations"]["dice"]["changes_summary"])


100%|██████████| 1/1 [00:17<00:00, 17.79s/it]

Prediction:genuine
Fraud Probability: 0.00012170408890489896
Confidence: 0.9998782959110951
[{'feature': 'category', 'shap_score': 0.479104750527541, 'lime_score': 0.9090831945278702, 'ensemble_expl_score': 0.6510961281276727}, {'feature': 'amt', 'shap_score': 0.2838744999903566, 'lime_score': 0.09091680293897698, 'ensemble_expl_score': 0.2066914211698047}, {'feature': 'trans_hour', 'shap_score': 0.09888546732331961, 'lime_score': 0.0, 'ensemble_expl_score': 0.05933128039399176}, {'feature': 'city_pop', 'shap_score': 0.04275381719101087, 'lime_score': 0.0, 'ensemble_expl_score': 0.025652290314606523}, {'feature': 'age', 'shap_score': 0.03371231747797132, 'lime_score': 0.0, 'ensemble_expl_score': 0.020227390486782793}, {'feature': 'gender', 'shap_score': 0.02624291289244322, 'lime_score': 0.0, 'ensemble_expl_score': 0.015745747735465932}, {'feature': 'long', 'shap_score': 0.02221159217050877, 'lime_score': 0.0, 'ensemble_expl_score': 0.01332695530230526}, {'feature': 'merch_long', 'shap

Save artifacts

In [148]:
#  Save PyTorch MLP weights
mlp_payload = {
    "state_dict": mlp_model.state_dict(),
    "input_dim": int(X_train_processed.shape[1]),
    "feature_names": list(feature_names),
        "best_threshold": float(best_t)
}
torch.save(mlp_payload, nn_model_path)
print("Saved NN model to:", nn_model_path)

# Keep them small but representative
xai_bg_df = X_train_processed.sample(500, random_state=42).copy()
xai_bg_np = xai_bg_df.to_numpy()

# Separate small samples for each explainer type
bundle_xai_assets = {
    "shap_bg_df": xai_bg_df.sample(300, random_state=42).copy(),   
    "lime_train_np": xai_bg_np[:500],                              
    "anchor_bg_np": xai_bg_np[:500],                               
}
# Also store quantile value you used in permitted_range for "amt"
amt_q99 = float(X_train_fe["amt"].quantile(0.99))

# Small sample in RAW FE space for DiCE schema + dtype inference
dice_schema_df = X_train_fe.sample(1000, random_state=42).copy()
dice_schema_df["is_fraud"] = y_train.loc[dice_schema_df.index].values


# Save full sklearn + metadata bundle (for app.py)
model_bundle = {
    # preprocessing + feature engineering metadata
    "drop_cols": drop_cols,
    "categorical_cols": categorical_cols,
    "numeric_cols": numeric_cols,
    "preprocessor": preprocessor,
    "processed_feature_names": list(feature_names),
    "raw_fe_columns": list(X_train_fe.columns),

    # trained ML models
    "rf_model": rf_model,
    "xgb_model": xgb_model,
    "lgbm_model": lgbm_model,

    # ensemble settings
    "weights": {"rf": w_rf, "xgb": w_xgb, "lgbm": w_lgbm, "mlp": w_mlp},
    "best_threshold": float(best_t),

    # DiCE constraints 
    "permitted_range_base": {
        "amt": [0.0, amt_q99],
        "age": [18.0, 100.0],
        "trans_hour": [0.0, 23.0],
        "trans_month": [1.0, 12.0],
        "is_weekend": [0.0, 1.0],
    },
    "dice_features_to_vary": ["amt", "category", "trans_hour", "is_weekend"],
    "dice_schema_df": dice_schema_df,
    # XAI recreation assets
    "xai_assets": bundle_xai_assets
}

with open(model_bundle_path, "wb") as f:
    pickle.dump(model_bundle, f)

print("Saved model bundle to:", model_bundle_path)
print("Bundle keys:", list(model_bundle.keys()))


Saved NN model to: ../Artifacts/nn_model.pt
Saved model bundle to: ../Artifacts/model_bundle.pkl
Bundle keys: ['drop_cols', 'categorical_cols', 'numeric_cols', 'preprocessor', 'processed_feature_names', 'raw_fe_columns', 'rf_model', 'xgb_model', 'lgbm_model', 'weights', 'best_threshold', 'permitted_range_base', 'dice_features_to_vary', 'dice_schema_df', 'xai_assets']


In [2]:
#%pip install streamlit 
#%pip install pytest 
# to run the app use: py -m streamlit run .\UI\app.py 
